In [14]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()   # <-- uncomment and run ONCE on first use, then re-comment

  Activating project at `c:\JuliaProjects\More_zodi`
┌ Warning: The project dependencies or compat requirements have changed since the manifest was last resolved.
│ It is recommended to `Pkg.resolve()` or consider `Pkg.update()` if necessary.
└ @ Pkg.API C:\Users\marka\.julia\juliaup\julia-1.12.6+0.x64.w64.mingw32\share\julia\stdlib\v1.12\Pkg\src\API.jl:1322


Pkg.Types.PkgError: `CSV` is a direct dependency, but does not appear in the manifest. If you intend `CSV` to be a direct dependency, run `Pkg.resolve()` to populate the manifest. Otherwise, remove `CSV` with `Pkg.rm("CSV")`. Finally, run `Pkg.instantiate()` again.

In [ ]:
Pkg.resolve()

In [ ]:
import Pkg
Pkg.add("HDF5")

   Resolving package versions...
   Installed GR_jll ─────────────────── v0.73.27+0
   Installed LERC_jll ───────────────── v4.2.0+0
   Installed JpegTurbo_jll ──────────── v3.2.0+1
   Installed SciMLPublic ────────────── v1.3.0
   Installed GR ─────────────────────── v0.73.27
   Installed ImageIO ────────────────── v0.6.10
   Installed Unitful ────────────────── v1.29.0
   Installed Sixel ──────────────────── v0.1.6
   Installed InlineStrings ──────────── v1.4.6
   Installed NNlib ──────────────────── v0.9.45
   Installed PlotUtils ──────────────── v1.5.1
   Installed StaticArrays ───────────── v1.9.22
   Installed URIParser ──────────────── v0.4.1
   Installed Roots ──────────────────── v3.0.10
   Installed GPUArrays ──────────────── v11.5.16
   Installed BFloat16s ──────────────── v0.6.2
   Installed Cairo_jll ──────────────── v1.18.8+0
   Installed ZygoteRules ────────────── v0.2.8
   Installed Gamma ──────────────────── v1.2.0
   Installed FFMPEG ─────────────────── v0.4.6
   Inst

In [7]:
import Pkg; Pkg.add("StatsBase")

   Resolving package versions...
   Installed OrderedCollections ─ v2.0.2
   Installed Statistics ───────── v1.11.5
   Installed LogExpFunctions ──── v1.0.2
   Installed StatsBase ────────── v0.34.13
    Updating `C:\JuliaProjects\More_zodi\Project.toml`
  [2913bbd2] + StatsBase v0.34.13
    Updating `C:\JuliaProjects\More_zodi\Manifest.toml`
  [66dad0bd] + AliasTables v1.1.3
  [9a962f9c] + DataAPI v1.16.0
  [864edb3b] + DataStructures v0.19.6
  [ffbed154] + DocStringExtensions v0.9.5
  [92d709cd] + IrrationalConstants v0.2.6
  [2ab3a3ac] + LogExpFunctions v1.0.2
  [e1d29d7a] + Missings v1.2.0
  [bac558e1] + OrderedCollections v2.0.2
  [43287f4e] + PtrArrays v1.4.0
  [a2af1166] + SortingAlgorithms v1.2.3
  [10745b16] + Statistics v1.11.5
  [82ae8749] + StatsAPI v1.8.0
  [2913bbd2] + StatsBase v0.34.13
  [37e2e46d] + LinearAlgebra v1.12.0
  [9e88b42a] + Serialization v1.11.0
  [2f01184e] + SparseArrays v1.12.0
  [4536629a] + OpenBLAS_jll v0.3.29+0
  [bea87d4a] + SuiteSparse_jll v7.8.3+2

In [5]:
using HDF5, StatsBase, Statistics, Random, CairoMakie, Printf
using NaNStatistics, ProgressMeter, ColorSchemes
black_latexfonts = merge(theme_black(), theme_latexfonts())
set_theme!(black_latexfonts)
CairoMakie.disable_mime!("svg", "pdf", "text/html")

#Pulling in data files for the project.

DATA_DIR = joinpath(@__DIR__, "data")

pointing_file = joinpath(DATA_DIR, "boresight_pointing.h5")
vectors_file   = joinpath(DATA_DIR, "spherex_vectors_bore_interpolated.h5")
allsky_file   = joinpath(DATA_DIR, "spherex_zodipy_5um_boresight_healpix_components_interpolated.h5")
emission_file = joinpath(DATA_DIR, "spherex_zodipy_5um_boresight_interpolated.h5")

# Where is SPHEREx looking? 
f = h5open(pointing_file)
nvec_unit_cartesian   = read(f["pointing/nvec_unit_cartesian"])   
ecliptic_longitude_deg = read(f["pointing/ecliptic_longitude_deg"])
ecliptic_latitude_deg  = read(f["pointing/ecliptic_latitude_deg"])
times = read(f["times/julian_date"])   # Julian Date (days); see the Times note above
close(f)
size(nvec_unit_cartesian)


f = h5open(vectors_file)
earth_positions     = read(f["earth/positions"])
v_earth             = read(f["earth/velocities"])
jupiter_positions   = read(f["jupiter/positions"])
spherex_positions   = read(f["spherex/positions"])
v_spherex           = read(f["spherex/velocities"])
moon_positions      = read(f["moon/positions"])
close(f)

zhat_vec = zero(earth_positions); zhat_vec[3, :] .= 1.0
xhat_vec = zero(earth_positions); xhat_vec[1, :] .= 1.0
yhat_vec = zero(earth_positions); yhat_vec[2, :] .= 1.0

v_earth_unit        = v_earth   ./ sqrt.(sum(v_earth .^ 2, dims=1))
v_spherex_unit      = v_spherex ./ sqrt.(sum(v_spherex .^ 2, dims=1))
size(spherex_positions)

# --- The target: Kelsall-model 5 μm emission along each boresight ---
f = h5open(emission_file)
y = read(f["emission"])            # MJy/sr, length N
meta = attrs(f["metadata"])
println("description: ", meta["description"])
println("wavelength:  ", meta["wavelength"], " ", meta["wavelength_units"])
println("units:       ", meta["emission_units"])
close(f)
extrema(y)


dot_prod_matrix(x, y) = dropdims(sum(x .* y, dims=1), dims=1)

# make_vectors(nvec, idx) — THE one place the model inputs are defined.
#   `nvec`: 3 × M pointing directions.  `idx`: indices into the per-exposure ephemeris
#   columns — use 1:N for the boresight data, or fill(t, M) to evaluate everything at
#   a single epoch t (used for the all-sky map at the end).
# Add a vector here and every model and validation downstream sees it automatically.
make_vectors(nvec, idx) = [
    "n" => nvec,
    "S" => spherex_positions[:, idx],
    "z" => zhat_vec[:, idx],
    # candidate additions — uncomment to test (see the physics questions above
    # and stretch goal #2):
    "E"  => earth_positions[:, idx],
    "vS" => v_spherex_unit[:, idx],
    "vE" => v_earth_unit[:, idx],
    "J"  => jupiter_positions[:, idx],
    "x"  => xhat_vec[:, idx],        # fixed in-plane inertial axes — see stretch goal #2
    "y"  => yhat_vec[:, idx],
]

# all pairwise dot products among the registered vectors
function dot_features(vecs)
    names, cols = String[], Vector{Float64}[]
    for i in eachindex(vecs), j in i:length(vecs)
        push!(names, string(first(vecs[i]), "·", first(vecs[j])))
        push!(cols, dot_prod_matrix(last(vecs[i]), last(vecs[j])))
    end
    return names, reduce(hcat, cols)
end

# all squares and pairwise products of (already standardized) features —
# the hand-made curvature for the linear-plus-quadratic model
function quadratic_terms(names, D)
    qnames, qcols = String[], Vector{Float64}[]
    for i in axes(D, 2), j in i:size(D, 2)
        push!(qnames, i == j ? "($(names[i]))²" : "($(names[i]))($(names[j]))")
        push!(qcols, D[:, i] .* D[:, j])
    end
    return qnames, reduce(hcat, qcols)
end

# a first look at the raw dot products and their spread
dot_names_all, Fd_raw = dot_features(make_vectors(nvec_unit_cartesian, 1:length(y)))
for (nm, i) in zip(dot_names_all, axes(Fd_raw, 2))
    @printf("%-6s  median %10.3e   iqr %10.3e\n", nm, median(Fd_raw[:, i]), iqr(Fd_raw[:, i]))
end



time_days = @isdefined(time_days) ? time_days : (times .- times[1])

# ==============================================================================
# MASTER CLEAN SPLITS INITIALIZATION (CHRONOLOGICALLY ENFORCED)
# ==============================================================================

N = length(y)

# Create a mapping vector that represents the strictly chronological order of rows
c_idx = sortperm(time_days)

# 1. Chronological 75/25 (Train: Days 0.0 -> 112.5 | Test: Days 112.5 -> 150.0)
sp75 = floor(Int, 0.75 * N)
train_idx_75 = c_idx[1:sp75]
test_idx_25  = c_idx[(sp75 + 1):N]

# 2. Chronological 50/50 (Train: Days 0.0 -> 75.0  | Test: Days 75.0 -> 150.0)
sp50 = floor(Int, 0.50 * N)
train_idx_50 = c_idx[1:sp50]
test_idx_50  = c_idx[(sp50 + 1):N]

# 3. Chronological 25/75 (Train: Days 0.0 -> 37.5  | Test: Days 37.5 -> 150.0)
sp25 = floor(Int, 0.25 * N)
train_idx_25 = c_idx[1:sp25]
test_idx_75  = c_idx[(sp25 + 1):N]

# 4. Chronological 80/20 (Train: Days 0.0 -> 120.0 | Test: Days 120.0 -> 150.0)
sp80 = floor(Int, 0.80 * N)
train_idx_80 = c_idx[1:sp80]
test_idx_20  = c_idx[(sp80 + 1):N]

# 5. Chronological 85/15 (Train: Days 0.0 -> 127.5 | Test: Days 127.5 -> 150.0)
sp85 = floor(Int, 0.85 * N)
train_idx_85 = c_idx[1:sp85]
test_idx_15  = c_idx[(sp85 + 1):N]

# 6. Random 50/50 Split (Randomly sampled indices across all 150 days)
# (This remains unchanged since random is random regardless of order)
rng = Xoshiro(2026)
train_idx_rnm = sort(sample(rng, 1:N, N ÷ 2, replace=false))
test_idx_rnm  = setdiff(1:N, train_idx_rnm)

# 7. 30-Day Time Block Split (5 alternating 30-day blocks)
K_block = 5
block_sz = N ÷ K_block
train_idx_block = Int[]
test_idx_block  = Int[]
for k in 1:K_block
    # Slice the chronological index array instead of just 1:N
    blk_range = ((k-1)*block_sz + 1):((k == K_block) ? N : k*block_sz)
    blk = c_idx[blk_range]
    isodd(k) ? append!(train_idx_block, blk) : append!(test_idx_block, blk)
end

# 8. 7-Day Alternating Week Split (21 alternating 7-day blocks)
num_wk_blocks = ceil(Int, 150.0 / 7.0)
wk_sz = N ÷ num_wk_blocks
train_idx_week = Int[]
test_idx_week  = Int[]
for k in 1:num_wk_blocks
    blk_range = ((k-1)*wk_sz + 1):((k == num_wk_blocks) ? N : k*wk_sz)
    blk = c_idx[blk_range]
    isodd(k) ? append!(train_idx_week, blk) : append!(test_idx_week, blk)
end

# 9. Mid-Mission Holdout 50% (Outer 50% Train, Middle 50% Test)
p1 = floor(Int, 0.25 * N)
p2 = floor(Int, 0.75 * N)
test_idx_mid  = c_idx[(p1 + 1):p2]
train_idx_mid = vcat(c_idx[1:p1], c_idx[(p2 + 1):N])

# Register all splits cleanly into Master Dictionary
splits = Dict(
    "75/25 Chronological" => (train_idx_75, test_idx_25),
    "50/50 Chronological" => (train_idx_50, test_idx_50),
    "25/75 Chronological" => (train_idx_25, test_idx_75),
    "80/20 Chronological" => (train_idx_80, test_idx_20),
    "85/15 Chronological" => (train_idx_85, test_idx_15),
    "random split"        => (train_idx_rnm, test_idx_rnm),
    "30 day split"        => (train_idx_block, test_idx_block),
    "7-day alternating"   => (train_idx_week, test_idx_week),
    "50/50 mid-mission"   => (train_idx_mid, test_idx_mid)
)

println("All splits successfully initialized with strict chronological mapping!")

ArgumentError: ArgumentError: Package StatsBase not found in current path.
- Run `import Pkg; Pkg.add("StatsBase")` to install the StatsBase package.